In [1]:
import torch 
import torch.nn as nn
import numpy as np 
from datasets import load_dataset
from tqdm import tqdm 
import yaml 

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import importlib

import robustness.audio_functions.audio_transforms as at 
from lightning_scripts.lightning_ssl import LitAudioSSL 
import robustness.audio_models as architectures



In [2]:
config_path = "model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

ckpt_path = "model_checkpoints/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment/checkpoints/epoch=160-step=28980-best_word_task.ckpt"


## Get just the cochleagram and feature extractor 

In [3]:
module = LitAudioSSL.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False).eval()
audio_transforms = module.transforms 
feature_extractor = module.model
# model = torch.nn.ModuleDict({'front_end':feature_extractor.front_end, 'f':feature_extractor.model.f})



# Get the dataset 

In [4]:
speech_commands = load_dataset("google/speech_commands", 'v0.01')

In [6]:
train_split = speech_commands['train']
# remove silence 
train_split = train_split.filter(lambda example: (not ('_silence_' in example['file'])) and (not (example['audio']['array'] is None)))
train_split = train_split.shuffle(seed=1)
val_split = speech_commands['validation']
test_split = speech_commands['test']

In [7]:
dataloader =  torch.utils.data.DataLoader(train_split, batch_size=2) 

In [8]:
for batch in dataloader:
    break

In [9]:
batch

{'file': ['two/d98f6043_nohash_0.wav', 'bird/ab3f0c1b_nohash_0.wav'],
 'audio': {'path': ['two/d98f6043_nohash_0.wav', 'bird/ab3f0c1b_nohash_0.wav'],
  'array': tensor([[-3.0518e-05, -6.1035e-05, -9.1553e-05,  ..., -6.1035e-05,
           -3.0518e-05, -6.1035e-05],
          [ 7.3547e-03,  7.7820e-03,  8.0872e-03,  ..., -8.1787e-03,
           -8.6365e-03, -9.0637e-03]], dtype=torch.float64),
  'sampling_rate': tensor([16000, 16000])},
 'label': tensor([12, 21]),
 'is_unknown': tensor([True, True]),
 'speaker_id': ['d98f6043', 'ab3f0c1b'],
 'utterance_id': tensor([0, 0])}

## Write quick training loop

In [7]:
from torchaudio.transforms import Resample
resamp = Resample(16_000, 20_000)

In [8]:
ix = 0 
batch_size = 10 
start = ix * batch_size 
end = start + batch_size 
batch = train_split[start:end]
# audio = batch['audio']
# word_int_label = batch['label']


class CenterCropOrPad:
    def __init__(self, sig_length):
        self.sig_length = sig_length

    def __call__(self, x):
        if x.shape[0] < self.sig_length:
        # edge pad if x is too short 
            pad_dur = (self.sig_length - len(x)) // 2 + 1 
            # print(f"X shape before pad: {x.shape}")
            x = nn.functional.pad(x, (pad_dur, pad_dur), "constant", 0 )
            # print(f"X shape after pad: {x.shape}")
        # re-compute crop bound

        # else:        
        start_idx = int((x.shape[0] - self.sig_length)/2)
        x = x[start_idx:start_idx+self.sig_length]

        return x

crop_or_pad = CenterCropOrPad(40000)
db_spl = 60 
set_dbSPL = at.DBSPLNormalizeForegroundAndBackground(db_spl)

def process_batch(batch):
    audio = batch['audio']
    word_int_label = torch.tensor(batch['label']).cuda()
    label_ixs = []
    tformed_audio = []
    for ix, eg in enumerate(audio):
        wav = torch.from_numpy(eg['array'])
        # wav, _ = audio_transforms(eg['array'], None)
        wav = resamp(wav.float())
        # zero pad word to middle of frame 
        wav = crop_or_pad(wav.squeeze())
        wav, _ = set_dbSPL(wav, None)
        if wav is None:
            continue
        tformed_audio.append(wav.unsqueeze(0))
        label_ixs.append(ix)
    audio = torch.stack(tformed_audio).cuda()
    word_int_label = word_int_label[label_ixs]
    return audio, word_int_label

audio, labels = process_batch(batch)

In [9]:
audio.shape

torch.Size([10, 1, 40000])

In [10]:
### Set up class wrapper
import torch.nn as nn 
class TransferWrapper(nn.Module):
    def __init__(self, feature_extractor, num_classes, layer_out):
        super().__init__()
        self.layer_out = layer_out
        self.feature_extractor = feature_extractor.eval()
        if config['model']['arch_kwargs']['backbone'] == 'kell2018':
            layer_size_dict = {'avgpool':512*9*5,
                               'relufc':4096}
        proj_out_dim = layer_size_dict[layer_out]
        self.classifier = nn.Linear(proj_out_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            predictions, rep, all_outputs = self.feature_extractor(x, with_latent=True, fake_relu=True)
            activations = all_outputs[self.layer_out]
            activations = activations.detach()
        logits = self.classifier(activations)
        return logits 




In [11]:
n_classes = len(np.unique(train_split['label']))
transfer_model = TransferWrapper(feature_extractor, n_classes, 'relufc' ).cuda()

### Dummy training loop

In [12]:
from audio_ssl.misc import CosineWarmupScheduler


In [14]:

batch_size = 64
n_batches = len(train_split) // batch_size 





best_loss = 10.
lr = 0.1
optimizer = torch.optim.AdamW(transfer_model.classifier.parameters(), lr=lr)
lr_scheduler = CosineWarmupScheduler(
            optimizer=optimizer,
            batch_size=batch_size, # is global batch size
            warmup_steps=0,
            max_steps=n_batches,
            lr=lr
        )
        
loss_fn = nn.CrossEntropyLoss() 

for ix in (pbar := tqdm(range(n_batches))):
    start = ix * batch_size 
    end = start + batch_size 
    batch = train_split[start:end]
    audio, labels = process_batch(batch)

    optimizer.zero_grad()
    logits = transfer_model(audio)
    loss = loss_fn(logits, labels)
    loss.backward()
    optimizer.step()
    lr_scheduler.step()
    if loss.item() < best_loss:
        best_loss = loss.detach().item()

    pbar.set_postfix_str(f"train loss: {loss.detach().item():.4f} best loss: {best_loss:.4f}")#



/mnt/home/igriffith/envs/cochdnn_huggingface/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
  3%|▎         | 23/798 [00:14<07:57,  1.62it/s, train loss: 1.4906 best loss: 1.4357]

In [18]:
### Demo val loop:

n_batches = len(val_split) // batch_size 
all_accs = []
for ix in (pbar := tqdm(range(n_batches))):
    start = ix * batch_size 
    end = start + batch_size 
    batch = val_split[start:end]
    audio, labels = process_batch(batch)

    with torch.no_grad():
        logits = transfer_model(audio)
        loss = loss_fn(logits, labels)

    model_preds = logits.softmax(-1).argmax(-1)
    accuracy = (model_preds == labels).float().mean()
    all_accs.append(accuracy.item())
    if loss.item() < best_loss:
        best_loss = loss.detach().item()
    pbar.set_postfix_str(f"val loss: {loss.detach().item():.4f} val acc: {accuracy:.4f}")#
np.mean(all_accs)


  0%|          | 0/106 [00:00<?, ?it/s]

 69%|██████▉   | 73/106 [00:45<00:20,  1.60it/s, val loss: 0.5330 val acc: 0.8750]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Try lightning module 

In [4]:
import lightning_scripts.eval_speech_commands_transfer as sc_transfer
import lightning as L

importlib.reload(sc_transfer)
SSLClassifier = sc_transfer.SSLClassifier

In [14]:
## update config for params 
config_path = "model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

ckpt_path = "model_checkpoints/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment/checkpoints/epoch=160-step=28980-best_word_task.ckpt"


## update config for params 
config['hparas']['batch_size'] = 64
config['num_workers'] = 4
config['num_gpus'] = 1
config['hparas']['optimizer'] = "AdamW"
config['hparas']['lr'] = 0.01
config['hparas']['lr_schedule'] = True
config['hparas']['num_warmup_steps_or_ratio'] = 0

importlib.reload(sc_transfer)
SSLClassifier = sc_transfer.SSLClassifier
module = SSLClassifier(config=config, ckpt_path=ckpt_path, layer_out='relufc')


print(len(module.train_dataloader()))

trainer = L.Trainer(devices=1, max_epochs=3) 

trainer.fit(module)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


799



  | Name              | Type                                  | Params | Mode 
------------------------------------------------------------------------------------
0 | feature_extractor | OptimizedModule                       | 116 M  | eval 
1 | set_dbSPL         | DBSPLNormalizeForegroundAndBackground | 0      | train
2 | classifier        | Linear                                | 122 K  | train
3 | loss_fn           | CrossEntropyLoss                      | 0      | train
4 | accuracy          | MulticlassAccuracy                    | 0      | train
------------------------------------------------------------------------------------
122 K     Trainable params
116 M     Non-trainable params
116 M     Total params
466.389   Total estimated model params size (MB)
4         Modules in train mode
54        Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

: 

: 

: 

In [2]:
print("Running inference")
test_dataloader = module.test_dataloader()
outputs = trainer.predict(module, test_dataloader, return_predictions=True)

Running inference


NameError: name 'module' is not defined

In [1]:
top1_word = []
top5_word = []

for record in outputs:
    top1_word.append(record['top1'])
    top5_word.append(record['top5'])
n_examples = len(outputs)


output_dict = {
    "word_top1_mean": torch.stack(top1_word).mean(),
    "word_top1_sem": torch.stack(top1_word).std() / np.sqrt(n_examples),
    "word_top5_mean": torch.stack(top5_word).mean(),
    "word_top5_sem": torch.stack(top5_word).std() / np.sqrt(n_examples),
}
output_dict 
    

NameError: name 'outputs' is not defined